# 🏗️ Bronze Layer — Ingestion sirovih podataka

## Projekat: Used Vehicles Data Platform
**Notebook:** `01_bronze_ingestion`  
**Layer:** Bronze (Raw Data)  
**Autor:** Petra Petronijević  
**Poslednja izmena:** 2026-04-29

---

## Šta ovaj notebook radi?

Ovaj notebook učitava **sirove podatke** iz Azure Data Lake Storage (ADLS) i čuva ih kao **Delta tabele** u Databricks Unity Catalog-u.

### Zašto je ovo važno?
- ✅ **Nezavisnost od ADLS-a** — Jednom kada su podaci sačuvani kao Delta tabele, više NE zavise od Azure subscription-a
- ✅ **Medallion arhitektura** — Bronze → Silver → Gold (industrija standard za data engineering)
- ✅ **Ponovljivost** — Ovaj notebook se može ponovo pokrenuti kad god stignu novi podaci
- ✅ **Audit trail** — Svaki red ima metadata (kada je učitan, iz kog fajla)

### Medallion arhitektura (ukratko):
| Layer | Opis | Stanje podataka |
|-------|------|------------------|
| 🥉 **Bronze** | Sirovi podaci "as-is" | Neočišćeni, originalni formati |
| 🥈 **Silver** | Očišćeni i standardizovani | Ispravni tipovi, bez duplikata |
| 🥇 **Gold** | Business-ready agregacije | KPI-evi, reporti, ML features |

### Izvori podataka:
| # | Izvor | Format | Opis |
|---|-------|--------|------|
| 1 | FirstSourceSystem | CSV | 111k+ redova, svi tipovi STRING |
| 2 | SecondSourceSystem | Parquet | Optimizovan format, ispravni tipovi |
| 3 | ThirdSourceSystem | Avro | Streaming-friendly format |
| 4 | FourthSourceSystem | JSON | Tekstualni strukturirani format |
| 5 | model_to_brand_lookup | CSV | Lookup tabela: model → brend |
| 6 | ServiceBook | CSV | Istorija servisa vozila |

## ⚙️ 1. Konfiguracija
Definisanje konstanti i putanja — ako se bilo šta promeni (npr. novi storage account), menjaš SAMO ovde.

In [0]:
# ==============================================================================
# KONFIGURACIJA - Sve promenljive na jednom mestu
# ==============================================================================
# Ako se promeni storage account ili putanja, izmeni SAMO ovde.
# Ostatak notebook-a koristi ove promenljive.
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# --- Destinacija za Bronze tabele ---
# Koristimo hive_metastore jer ima Databricks-managed storage (DBFS).
# To znači da podaci NE ZAVISE od Azure ADLS subscription-a!
# Jednom sačuvani ovde, ostaju dostupni zauvek.
CATALOG = "hive_metastore"
SCHEMA = "bronze"

# --- ADLS putanje (Azure Data Lake Storage) ---
# NAPOMENA: Ove putanje rade SAMO dok je Azure subscription aktivna.
# Zato ih čuvamo u Delta tabele — jednom sačuvano, zauvek dostupno!
ADLS_BASE = "abfss://raw@usedvehicledl.dfs.core.windows.net"

# --- Definicija svih izvora podataka ---
# Svaki izvor ima: ime tabele, putanja, format fajla, i opcije za čitanje
SOURCE_CONFIGS = [
    {
        "table_name": "first_source_system",
        "path": f"{ADLS_BASE}/source_systems/FirstSourceSystem",
        "format": "csv",
        "options": {"header": "true", "inferSchema": "true"}
    },
    {
        "table_name": "second_source_system",
        "path": f"{ADLS_BASE}/source_systems/SecondSourceSystem",
        "format": "parquet",
        "options": {}
    },
    {
        "table_name": "third_source_system",
        "path": f"{ADLS_BASE}/source_systems/ThirdSourceSystem",
        "format": "avro",
        "options": {}
    },
    {
        "table_name": "fourth_source_system",
        "path": f"{ADLS_BASE}/source_systems/FourthSourceSystem",
        "format": "json",
        "options": {"multiLine": "false"}
    },
    {
        "table_name": "model_to_brand_lookup",
        "path": f"{ADLS_BASE}/lookup/model_to_brand_lookup",
        "format": "csv",
        "options": {"header": "true", "inferSchema": "true"}
    },
    {
        "table_name": "service_book",
        "path": f"{ADLS_BASE}/service_book/ServiceBook",
        "format": "csv",
        "options": {"header": "true", "inferSchema": "true"}
    },
]

print(f"✅ Konfiguracija učitana.")
print(f"   Catalog: {CATALOG}")
print(f"   Schema:  {SCHEMA}")
print(f"   Broj izvora: {len(SOURCE_CONFIGS)}")
print(f"   🛡️ Storage: Databricks-managed (nezavisno od Azure ADLS!)")

## 🗄️ 2. Kreiranje Bronze scheme
Kreiramo `bronze` schemu u Unity Catalog-u. Tu će živeti svi sirovi podaci kao Delta tabele.

In [0]:
# ==============================================================================
# KREIRANJE BRONZE SCHEME
# ==============================================================================
# CREATE SCHEMA IF NOT EXISTS — bezbedno za ponavljanje (idempotentno).
# Ako schema već postoji, ništa se neće desiti.
# ==============================================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

print(f"✅ Schema '{CATALOG}.{SCHEMA}' je spremna.")
print(f"   Svi podaci će biti sačuvani kao Delta tabele u ovoj schemi.")
print(f"   🛡️ Delta tabele su NEZAVISNE od Azure ADLS storage-a!")

## 🚀 3. Funkcija za ingestion
Definisemo jednu reusable funkciju koja:
1. Čita podatke iz ADLS-a (bilo koji format)
2. Dodaje metadata kolone (`_ingestion_timestamp`, `_source_file_name`)
3. Čuva kao Delta tabelu u Bronze schemi

> 💡 **Best practice:** Jedna funkcija za sve izvore = manje koda, manje grešaka, lakše održavanje.

In [0]:
# ==============================================================================
# FUNKCIJA ZA INGESTION
# ==============================================================================
# Ova funkcija je srce notebook-a. Prihvata konfiguraciju jednog izvora
# i obavlja kompletan ingestion pipeline:
#   1. Čita raw podatke iz ADLS-a
#   2. Dodaje metadata kolone (audit trail)
#   3. Piše kao Delta tabelu u Bronze schemu
# ==============================================================================

def ingest_to_bronze(source_config: dict) -> dict:
    """
    Učitava jedan izvor podataka iz ADLS-a i čuva ga kao Delta tabelu.
    
    Parametri:
        source_config (dict): Rečnik sa ključevima:
            - table_name: ime ciljne tabele
            - path: ADLS putanja do fajla
            - format: format fajla (csv, parquet, avro, json)
            - options: dodatne opcije za čitanje (header, inferSchema, itd.)
    
    Vraća:
        dict: Rezultat sa brojem redova i statusom
    """
    table_name = source_config["table_name"]
    path = source_config["path"]
    file_format = source_config["format"]
    options = source_config["options"]
    
    # Puno ime tabele (catalog.schema.table)
    full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    
    print(f"\n{'='*60}")
    print(f"⏳ Učitavanje: {table_name}")
    print(f"   Format: {file_format.upper()}")
    print(f"   Izvor:  {path}")
    print(f"   Cilj:   {full_table_name}")
    print(f"{'='*60}")
    
    try:
        # --- Korak 1: Čitanje podataka iz ADLS-a ---
        reader = spark.read.format(file_format)
        
        # Dodajemo opcije (svaki format ima svoje specifičnosti)
        for key, value in options.items():
            reader = reader.option(key, value)
        
        df = reader.load(path)
        
        # --- Korak 2: Dodavanje metadata kolona ---
        # Ove kolone pomažu za audit/tracking:
        #   - _ingestion_timestamp: KADA su podaci učitani
        #   - _source_path: IZ KOG IZVORA su došli (ADLS putanja)
        df = df.withColumn("_ingestion_timestamp", F.current_timestamp()) \
               .withColumn("_source_path", F.lit(path))
        
        # --- Korak 3: Čuvanje kao Delta tabela ---
        # mode="overwrite" — svaki put pišemo sve iznova (full refresh)
        # overwriteSchema=true — ako se schema promeni, ažurira se
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(full_table_name)
        
        row_count = spark.table(full_table_name).count()
        
        print(f"   ✅ USPEŠNO! Redova: {row_count:,}")
        
        return {"table": table_name, "rows": row_count, "status": "✅ OK"}
    
    except Exception as e:
        print(f"   ❌ GREŠKA: {str(e)[:200]}")
        return {"table": table_name, "rows": 0, "status": f"❌ {str(e)[:80]}"}

## ▶️ 4. Pokretanje ingestion-a
Učitavamo SVIH 6 izvora podataka i čuvamo ih kao Delta tabele.  

> ⚠️ **Posle ovog koraka, podaci su TRAJNO sačuvani u Databricks-u** — čak i ako Azure subscription istekne, Delta tabele ostaju dostupne!

In [0]:
# ==============================================================================
# POKRETANJE INGESTION-A ZA SVE IZVORE
# ==============================================================================
# Petlja prolazi kroz svaki izvor i poziva našu ingestion funkciju.
# Rezultati se čuvaju u listi za završni izveštaj.
# ==============================================================================

print("🚀 Počinjem ingestion svih izvora u Bronze layer...")
print(f"   Vreme početka: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Cilj: {CATALOG}.{SCHEMA}.*")

# Lista za čuvanje rezultata svake ingestion operacije
results = []

# Iteriramo kroz sve definisane izvore
for config in SOURCE_CONFIGS:
    result = ingest_to_bronze(config)
    results.append(result)

print(f"\n{'='*60}")
print(f"🏁 Ingestion završen!")
print(f"   Vreme završetka: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")

## ✅ 5. Verifikacija
Provera da su svi podaci uspešno učitani. Prikazujemo:
- Broj redova po tabeli
- Ukupan broj redova u celom Bronze layer-u
- Sample podataka iz svake tabele

In [0]:
# ==============================================================================
# SUMRANI IZVEŠTAJ - Pregled svih učitanih tabela
# ==============================================================================
# Prikazujemo rezultate ingestion-a kao tabelu za laku proveru.
# ==============================================================================

# Kreiramo DataFrame od rezultata za lep prikaz
df_results = spark.createDataFrame(results)

print("\n📊 SUMARNI IZVEŠTAJ INGESTION-A:")
print("=" * 60)

display(df_results)

# Ukupan broj redova
total_rows = sum(r["rows"] for r in results)
print(f"\n📌 UKUPNO REDOVA U BRONZE LAYER-U: {total_rows:,}")
print(f"📌 BROJ TABELA: {len(results)}")
print(f"\n🛡️ Svi podaci su sada TRAJNO sačuvani kao Delta tabele.")
print(f"   Više ne zavise od Azure ADLS storage-a!")
print(f"   Možeš ih koristiti sa: SELECT * FROM {CATALOG}.{SCHEMA}.<ime_tabele>")

In [0]:
# ==============================================================================
# PREVIEW - Prikaz prvih 3 reda iz svake tabele
# ==============================================================================
# Brza vizuelna provera da podaci izgledaju ispravno.
# ==============================================================================

for config in SOURCE_CONFIGS:
    table_name = config["table_name"]
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    
    print(f"\n{'='*60}")
    print(f"🔍 Preview: {full_name}")
    print(f"{'='*60}")
    
    display(spark.table(full_name).limit(3))

## 📌 6. Registracija u Unity Catalog
Ovaj korak kopira Bronze tabele iz `hive_metastore` u `used_vehicle_databricks_ws` katalog.

**Zašto?** Da biš mogla da vidiš tabele u Catalog Explorer-u pod svojim katalogom.

Tabele se čuvaju kao **EXTERNAL** tabele sa eksplicitnom LOCATION na ADLS-u — tako zaobilazimo problem sa managed storage-om ovog kataloga.

In [0]:
# ==============================================================================
# REGISTRACIJA U UNITY CATALOG (used_vehicle_databricks_ws)
# ==============================================================================
# Kreiramo EXTERNAL tabele u used_vehicle_databricks_ws.bronze katalogu
# sa eksplicitnim LOCATION-om na ADLS-u.
# Tako tabele postaju vidljive u Catalog Explorer-u.
# ==============================================================================

TARGET_CATALOG = "used_vehicle_databricks_ws"
TARGET_SCHEMA = "bronze"

# Kreiramo bronze schemu u ciljnom katalogu (ako ne postoji)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")

print(f"🚀 Registrujem tabele u {TARGET_CATALOG}.{TARGET_SCHEMA}...\n")

for config in SOURCE_CONFIGS:
    table_name = config["table_name"]
    location = f"{ADLS_BASE}/bronze/{table_name}"
    
    print(f"  ⏳ {table_name} → {TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}")
    
    try:
        spark.sql(f"""
            CREATE OR REPLACE TABLE {TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}
            LOCATION '{location}'
            AS SELECT * FROM {CATALOG}.{SCHEMA}.{table_name}
        """)
        print(f"     ✅ OK")
    except Exception as e:
        print(f"     ❌ {str(e)[:150]}")

print(f"\n🏁 Registracija završena!")
print(f"   Tabele su sada vidljive u: Catalog Explorer → {TARGET_CATALOG} → {TARGET_SCHEMA}")

In [0]:
# ==============================================================================
# VERIFIKACIJA - Provera da su tabele vidljive u Unity Catalog-u
# ==============================================================================

print(f"🔍 Tabele u {TARGET_CATALOG}.{TARGET_SCHEMA}:\n")
spark.sql(f"SHOW TABLES IN {TARGET_CATALOG}.{TARGET_SCHEMA}").show(truncate=False)

---
## 🚦 Sledeći koraci

✅ **Urađeno:** Bronze layer je spreman! Svi sirovi podaci su u Delta tabelama.

👉 **Sledeći notebook:** `02_silver_transformations` — gde ćemo:
- Standardizovati tipove podataka (sve STRING kolone iz Source 1 → ispravni tipovi)
- Parsirati različite formate datuma (`posting_date`)
- Ukloniti/obraditi NULL vrednosti i duplikate
- Spojiti sva 4 source sistema u jednu čistu tabelu
- Primeniti brand/model mapping iz lookup tabele
- Dodati currency conversion kolone (GBP, USD, AUD)
- Integrisati dnevne kurseve valuta preko javnog API-ja

👉 **Treći notebook:** `03_gold_layer` — gde ćemo:
- Dizajnirati Star/Snowflake schema dimenzionalni model
- Izračunati Reliability Score (0-100) za svako vozilo iz ServiceBook-a
- Napraviti ranking najboljih vozila po ceni za svaki make/model

---

### 📝 Napomene:
- Ovaj notebook je **idempotentan** — možeš ga pokrenuti koliko god puta želiš bez posledica
- Delta tabele podržavaju **Time Travel** — možeš videti prethodne verzije podataka
- Za proveru u SQL editoru: `SELECT * FROM hive_metastore.bronze.<ime_tabele>`
- Za pregledanje tabela: Data Explorer → used_vehicle_databricks_ws → bronze